In [ ]:
!pip install torch torchvision torchaudio --no-cache-dir

In [ ]:
!pip install torch-xla
!pip install opencv-python
!pip install numpy torchao
!pip install ninja
!pip install shap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.9/72.9 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.3 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ==============================================================
#  DDoS ResNet50 SHAP Analysis - All Multiplier Configurations
#  WITH ROHNAS ANALYTICAL ENERGY MODEL
#  WITH ACTUAL ACTIVATION INPUT TRACKING
# ==============================================================

import warnings
warnings.filterwarnings('ignore')

import os
import pickle
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import shap
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from scipy.stats import spearmanr
import gc
import math
import time

print("="*70)
print("DDoS ResNet50 SHAP Analysis - ALL MULTIPLIER CONFIGURATIONS")
print("WITH ROHNAS ANALYTICAL ENERGY MODEL")
print("WITH ACTUAL ACTIVATION INPUT TRACKING")
print("="*70)

# =============================================================================
# CONFIGURATION
# =============================================================================
MODEL_PATH = 'ddos_resnet50_best_model.pth'
MODEL_INFO_PATH = 'ddos_resnet50_model_info.json'
SCALER_PATH = 'ddos_feature_scaler.pkl'
DATASET_PATH = '/content/DDOS_attack_benign_dns.csv'
SAMPLE_IDX = 0

# =============================================================================
# MAC CONFIGURATIONS (from MIRAI code)
# =============================================================================
MAC_CONFIGURATIONS = {
    'mul8s_1KV6_add16se_2TN': {'multiplier': 'mul8s_1KV6', 'adder': 'add16se_2TN', 'mul_power_mW': 0.425, 'mul_delay_ns': 1.48, 'add_power_mW': 0.072, 'add_delay_ns': 0.50, 'mae': 0.00, 'mre': 0.00, 'level': 0},
    'mul8s_1KV8_add16se_2TN': {'multiplier': 'mul8s_1KV8', 'adder': 'add16se_2TN', 'mul_power_mW': 0.422, 'mul_delay_ns': 1.48, 'add_power_mW': 0.072, 'add_delay_ns': 0.50, 'mae': 0.0018, 'mre': 0.28, 'level': 1},
    'mul8s_1KV9_add16se_2TN': {'multiplier': 'mul8s_1KV9', 'adder': 'add16se_2TN', 'mul_power_mW': 0.410, 'mul_delay_ns': 1.47, 'add_power_mW': 0.072, 'add_delay_ns': 0.50, 'mae': 0.0064, 'mre': 0.90, 'level': 2},
    'mul8s_1KVP_add16se_2TN': {'multiplier': 'mul8s_1KVP', 'adder': 'add16se_2TN', 'mul_power_mW': 0.363, 'mul_delay_ns': 1.37, 'add_power_mW': 0.072, 'add_delay_ns': 0.50, 'mae': 0.051, 'mre': 2.73, 'level': 3},
    'mul8s_1L2J_add16se_2TN': {'multiplier': 'mul8s_1L2J', 'adder': 'add16se_2TN', 'mul_power_mW': 0.301, 'mul_delay_ns': 1.36, 'add_power_mW': 0.072, 'add_delay_ns': 0.50, 'mae': 0.081, 'mre': 4.41, 'level': 4},
    'mul8s_1L2L_add16se_2TN': {'multiplier': 'mul8s_1L2L', 'adder': 'add16se_2TN', 'mul_power_mW': 0.200, 'mul_delay_ns': 1.14, 'add_power_mW': 0.072, 'add_delay_ns': 0.50, 'mae': 0.23, 'mre': 12.26, 'level': 5},
    'mul8s_1L2N_add16se_2TN': {'multiplier': 'mul8s_1L2N', 'adder': 'add16se_2TN', 'mul_power_mW': 0.126, 'mul_delay_ns': 0.94, 'add_power_mW': 0.072, 'add_delay_ns': 0.50, 'mae': 0.52, 'mre': 27.44, 'level': 6},
    'mul8s_1L12_add16se_2TN': {'multiplier': 'mul8s_1L12', 'adder': 'add16se_2TN', 'mul_power_mW': 0.052, 'mul_delay_ns': 0.89, 'add_power_mW': 0.072, 'add_delay_ns': 0.50, 'mae': 3.08, 'mre': 135.77, 'level': 7}
}

# =============================================================================
# ROHNAS HARDWARE PARAMETERS (TPU Systolic Array Configuration)
# =============================================================================
HARDWARE_CONFIG = {
    'clock_period_ns': 3.0,
    'pe_array_size': 65536,
    'load_weights_cycles': 100,
    'sram_energy_pJ_per_access': 1.92,
}

# =============================================================================
# ACTIVATION TRACKER - STORES ACTUAL TENSORS
# =============================================================================
class ActivationTracker:
    """Track ACTUAL activation input tensors for each layer during forward pass"""

    def __init__(self):
        self.activations = {}  # Will store actual tensors
        self.hooks = []

    def register_hooks(self, model):
        """Register hooks to capture ACTUAL activation inputs"""
        def hook_fn(name):
            def hook(module, input, output):
                if isinstance(input, tuple):
                    input_tensor = input[0]
                else:
                    input_tensor = input

                # Store the ACTUAL input tensor (detached and moved to CPU)
                self.activations[name] = input_tensor.detach().cpu().clone()
            return hook

        # Register hooks for Conv2d and Linear layers
        for name, module in model.named_modules():
            if isinstance(module, (nn.Conv2d, nn.Linear, ApproxConv2d)):
                hook = module.register_forward_hook(hook_fn(name))
                self.hooks.append(hook)

    def remove_hooks(self):
        """Remove all hooks"""
        for hook in self.hooks:
            hook.remove()
        self.hooks = []

    def save_activations(self, filename):
        """Save actual activation tensors to file"""
        # Convert tensors to numpy for saving
        activation_data = {}
        for name, tensor in self.activations.items():
            activation_data[name] = tensor.numpy()

        with open(filename, 'wb') as f:
            pickle.dump(activation_data, f)

        print(f"    ✓ Saved {len(activation_data)} activation inputs to {filename}")
        return activation_data

    def get_summary(self, top_n=10):
        """Get summary statistics from actual tensors (for display purposes only)"""
        summary = []
        for name, tensor in self.activations.items():
            summary.append({
                'layer': name,
                'mean': tensor.mean().item(),
                'std': tensor.std().item(),
                'min': tensor.min().item(),
                'max': tensor.max().item(),
                'abs_mean': tensor.abs().mean().item(),
                'shape': tuple(tensor.shape)
            })

        # Sort by absolute mean (most active layers first)
        summary.sort(key=lambda x: x['abs_mean'], reverse=True)
        return summary[:top_n]

# =============================================================================
# APPROXIMATE CONV2D
# =============================================================================
class ApproxConv2d(nn.Module):
    """Enhanced approximate Conv2d layer with MRE-based quantization"""

    def __init__(self, in_channels, out_channels, kernel_size, stride=1,
                 padding=0, dilation=1, groups=1, bias=True, mre=0.0, axx_mult=None):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size if isinstance(kernel_size, tuple) else (kernel_size, kernel_size)
        self.stride = stride
        self.padding = padding
        self.dilation = dilation
        self.groups = groups
        self.mre = mre
        self.axx_mult = axx_mult

        self.weight = nn.Parameter(torch.empty(out_channels, in_channels // groups, *self.kernel_size))
        self.bias = nn.Parameter(torch.empty(out_channels)) if bias else None

        # Initialize
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))
        if bias:
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.weight)
            bound = 1 / math.sqrt(fan_in)
            nn.init.uniform_(self.bias, -bound, bound)

    def forward(self, x):
        if self.mre == 0 or self.mre < 0.01:
            # Exact computation
            out = F.conv2d(x, self.weight, self.bias, self.stride,
                          self.padding, self.dilation, self.groups)
        else:
            # Reduce precision based on MRE
            if self.mre < 0.5:
                scale = 128.0  # 8-bit
            elif self.mre < 1.5:
                scale = 64.0
            elif self.mre < 3.5:
                scale = 32.0
            elif self.mre < 8.0:
                scale = 16.0
            elif self.mre < 20.0:
                scale = 8.0
            elif self.mre < 50.0:
                scale = 4.0
            else:
                scale = 2.0

            # Quantize with clamping
            w_q = torch.clamp(self.weight * scale, -scale, scale - 1).round() / scale
            x_q = torch.clamp(x * scale, -scale, scale - 1).round() / scale

            # Compute
            out = F.conv2d(x_q, w_q, self.bias, self.stride,
                          self.padding, self.dilation, self.groups)

            # Add small controlled noise (deterministic based on values)
            if self.mre > 1.0:
                noise_magnitude = (self.mre / 100.0) * 0.05
                noise_seed_val = int((x.abs().mean().item() * 1000 + self.mre) % 10000)
                rng_state = torch.get_rng_state()
                torch.manual_seed(noise_seed_val)
                noise = torch.randn_like(out) * noise_magnitude
                torch.set_rng_state(rng_state)
                out = out + noise

        return out

# =============================================================================
# LAYER DESCRIPTOR FOR ROHNAS
# =============================================================================
class LayerDescriptor:
    """Layer descriptor compatible with RoHNAS analytical model"""

    def __init__(self, layer_type, name, input_shape, output_shape,
                 kernel_size=1, stride=1, params=None):
        self.layer_type = layer_type
        self.name = name
        self.input_shape = input_shape
        self.output_shape = output_shape
        self.kernel_size = kernel_size
        self.stride = stride
        self.params = params or {}

        # Parse shapes based on layer type
        if layer_type == 'conv':
            self.chin, self.hin, self.win = input_shape
            self.chout, self.hout, self.wout = output_shape
            self.nin = self.hin
            self.nout = self.hout
            self.capsin = 0
            self.capsout = 0
        elif layer_type == 'fc':
            self.chin = input_shape[0] if isinstance(input_shape, tuple) else input_shape
            self.chout = output_shape[0] if isinstance(output_shape, tuple) else output_shape
            self.nin = 1
            self.nout = 1
            self.hin = self.win = self.hout = self.wout = 1
            self.capsin = 0
            self.capsout = 0

    def get_weights_count(self):
        """wl - Number of weights (RoHNAS Eq. 7)"""
        if self.layer_type == 'conv':
            return self.kernel_size * self.kernel_size * self.chin * self.chout
        elif self.layer_type == 'fc':
            return self.chin * self.chout
        return 0

    def get_sum_count(self):
        """sl - Number of values to be summed (for RoHNAS Eq. 4)"""
        if self.layer_type == 'conv':
            return self.kernel_size * self.kernel_size * self.chin
        elif self.layer_type == 'fc':
            return self.chin
        return 0

    def get_feature_maps_count(self):
        """fl - Number of feature maps (for RoHNAS Eq. 5, 6)"""
        if self.layer_type == 'conv':
            return self.nout * self.nout
        elif self.layer_type == 'fc':
            return 1
        return 0

    def __repr__(self):
        if self.layer_type == 'conv':
            return (f"Conv({self.name}): [{self.chin}×{self.hin}×{self.win}] → "
                   f"[{self.chout}×{self.hout}×{self.wout}], k={self.kernel_size}")
        elif self.layer_type == 'fc':
            return (f"FC({self.name}): {self.chin} → {self.chout}")
        return f"{self.layer_type}({self.name})"

# =============================================================================
# LAYER PROFILER
# =============================================================================
class LayerProfiler:
    """Profile model layers by tracing a forward pass to get actual dimensions"""

    def __init__(self, model, input_shape=(1, 6)):
        self.model = model
        self.input_shape = input_shape
        self.layer_descriptors = []
        self.hooks = []

    def _register_hooks(self):
        """Register forward hooks to capture layer I/O shapes"""

        def hook_fn(name, layer_type):
            def hook(module, input, output):
                # Get input and output shapes
                if isinstance(input, tuple):
                    input = input[0]

                in_shape = tuple(input.shape[1:])  # Skip batch dimension
                out_shape = tuple(output.shape[1:])

                # Extract layer parameters
                if isinstance(module, (nn.Conv2d, ApproxConv2d)):
                    kernel_size = module.kernel_size[0] if isinstance(module.kernel_size, tuple) else module.kernel_size
                    stride = module.stride[0] if isinstance(module.stride, tuple) else module.stride

                    desc = LayerDescriptor(
                        layer_type='conv',
                        name=name,
                        input_shape=in_shape,
                        output_shape=out_shape,
                        kernel_size=kernel_size,
                        stride=stride,
                        params={
                            'in_channels': module.in_channels,
                            'out_channels': module.out_channels,
                            'groups': module.groups,
                            'padding': module.padding
                        }
                    )
                    self.layer_descriptors.append(desc)

                elif isinstance(module, nn.Linear):
                    desc = LayerDescriptor(
                        layer_type='fc',
                        name=name,
                        input_shape=(in_shape[0] if len(in_shape) == 1 else int(np.prod(in_shape)),),
                        output_shape=(out_shape[0],),
                        params={
                            'in_features': module.in_features,
                            'out_features': module.out_features
                        }
                    )
                    self.layer_descriptors.append(desc)

            return hook

        # Register hooks for Conv2d and Linear layers
        for name, module in self.model.named_modules():
            if isinstance(module, (nn.Conv2d, nn.Linear, ApproxConv2d)):
                layer_type = 'conv' if isinstance(module, (nn.Conv2d, ApproxConv2d)) else 'fc'
                hook = module.register_forward_hook(hook_fn(name, layer_type))
                self.hooks.append(hook)

    def profile(self):
        """Run a forward pass to extract layer information"""
        self._register_hooks()

        # Create dummy input
        dummy_input = torch.randn(*self.input_shape)

        # Forward pass
        self.model.eval()
        with torch.no_grad():
            _ = self.model(dummy_input)

        # Remove hooks
        for hook in self.hooks:
            hook.remove()

        return self.layer_descriptors

# =============================================================================
# ROHNAS ENERGY MODEL
# =============================================================================
class RoHNASEnergyModel:
    """RoHNAS analytical energy model with EvoApprox MAC energy integration"""

    def __init__(self, mac_config_name):
        config = MAC_CONFIGURATIONS[mac_config_name]
        self.config_name = mac_config_name
        self.approximation_level = config['level']

        # EvoApprox MAC energy parameters
        self.mul_energy_pJ = config['mul_power_mW']
        self.add_energy_pJ = config['add_power_mW']
        self.mac_latency_ns = config['mul_delay_ns'] + config['add_delay_ns']

        # Total MAC energy: E_MAC = E_mult + E_add
        self.mac_energy_pJ = self.mul_energy_pJ + self.add_energy_pJ
        self.mre = config['mre']

        # RoHNAS hardware configuration
        self.T = HARDWARE_CONFIG['clock_period_ns']
        self.pe_array_size = HARDWARE_CONFIG['pe_array_size']
        self.load_weights = HARDWARE_CONFIG['load_weights_cycles']
        self.E_memory = HARDWARE_CONFIG['sram_energy_pJ_per_access']

        print(f"\n{'='*80}")
        print(f"RoHNAS Energy Model with EvoApprox MAC: {self.config_name}")
        print(f"{'='*80}")
        print(f"Hardware Parameters:")
        print(f"  Clock Period (T):        {self.T} ns")
        print(f"  PE Array Size:           {self.pe_array_size}×{self.pe_array_size}")
        print(f"  Memory Energy (SRAM):    {self.E_memory} pJ/access (CACTI-P @ 45nm)")
        print(f"\nEvoApprox MAC Configuration:")
        print(f"  Approximation Level:     {self.approximation_level}")
        print(f"  MRE:                     {self.mre}%")
        print(f"  Multiplier Energy:       {self.mul_energy_pJ:.3f} pJ")
        print(f"  Adder Energy:            {self.add_energy_pJ:.3f} pJ")
        print(f"  Total MAC Energy (E_MAC): {self.mac_energy_pJ:.3f} pJ")
        print(f"\nEnergy Model Formula:")
        print(f"  E_total = Σ(m_acc/128 × E_memory) + Σ(cl × E_MAC)")
        print(f"  ↳ Replaced fixed PE power with EvoApprox MAC energy")
        print(f"{'='*80}\n")

    def calculate_layer_energy_rohnas(self, layer_desc):
        """Calculate energy for a single layer using RoHNAS analytical model"""
        # Get layer parameters
        wl = layer_desc.get_weights_count()
        sl = layer_desc.get_sum_count()
        fl = layer_desc.get_feature_maps_count()

        # Equation 4: Calculate wPEarray
        wPEarray = math.ceil(wl / (self.pe_array_size * min(self.pe_array_size, sl)))

        # Equation 5: Calculate m_acc
        if fl == 1:
            m_acc = 65536
        else:
            m_acc = self.pe_array_size * max(sl - 15, 1)

        # Equation 6: Calculate cl
        cl = wl * wPEarray + fl

        # Calculate latency
        latency_ns = cl * self.T

        # Updated Equation 7: Calculate energy
        memory_energy_pJ = (m_acc / 128.0) * self.E_memory
        pe_array_energy_pJ = cl * self.mac_energy_pJ

        total_energy_pJ = memory_energy_pJ + pe_array_energy_pJ

        # Memory footprint (assuming 4 bytes per weight - float32)
        memory_footprint_bytes = wl * 4

        return latency_ns, total_energy_pJ, memory_footprint_bytes

    def calculate_model_energy(self, layer_descriptors):
        """Calculate total model energy using RoHNAS analytical model"""
        total_latency_ns = 0.0
        total_energy_pJ = 0.0
        total_memory_bytes = 0.0
        layer_details = []

        for i, layer_desc in enumerate(layer_descriptors):
            latency_ns, energy_pJ, memory_bytes = self.calculate_layer_energy_rohnas(layer_desc)

            layer_details.append({
                'layer_idx': i,
                'layer_name': layer_desc.name,
                'layer_type': layer_desc.layer_type,
                'latency_ns': latency_ns,
                'energy_pJ': energy_pJ,
                'memory_bytes': memory_bytes,
                'wl': layer_desc.get_weights_count(),
                'sl': layer_desc.get_sum_count(),
                'fl': layer_desc.get_feature_maps_count()
            })

            total_latency_ns += latency_ns
            total_energy_pJ += energy_pJ
            total_memory_bytes += memory_bytes

        return {
            'total_latency_ns': total_latency_ns,
            'total_latency_ms': total_latency_ns / 1e6,
            'total_energy_pJ': total_energy_pJ,
            'total_energy_mJ': total_energy_pJ / 1e9,
            'total_memory_bytes': total_memory_bytes,
            'total_memory_MiB': total_memory_bytes / (1024 * 1024),
            'layer_details': layer_details,
            'config_name': self.config_name,
            'approximation_level': self.approximation_level
        }

    def calculate_shap_energy(self, n_features, n_samples, layer_descriptors):
        """Calculate energy for SHAP explanation computation"""
        # Calculate base model energy for one forward pass
        base_results = self.calculate_model_energy(layer_descriptors)

        # SHAP scaling factor
        forward_passes = n_features * n_samples * 2

        # Scale energy and latency
        shap_energy_pJ = base_results['total_energy_pJ'] * forward_passes
        shap_latency_ns = base_results['total_latency_ns'] * forward_passes

        return {
            'shap_energy_mJ': shap_energy_pJ / 1e9,
            'shap_energy_J': shap_energy_pJ / 1e12,
            'shap_latency_ms': shap_latency_ns / 1e6,
            'shap_latency_s': shap_latency_ns / 1e9,
            'forward_passes': forward_passes,
            'base_energy_per_pass_mJ': base_results['total_energy_mJ'],
            'base_latency_per_pass_ms': base_results['total_latency_ms'],
            'layer_details': base_results['layer_details'],
            'config_name': self.config_name,
            'approximation_level': self.approximation_level,
            'total_memory_MiB': base_results['total_memory_MiB']
        }

# =============================================================================
# MODEL DEFINITION
# =============================================================================
class EnhancedResNet50(nn.Module):
    def __init__(self, n_features, num_classes, use_se=True):
        super().__init__()
        self.n_features = n_features
        self.img_h = int(np.ceil(np.sqrt(n_features)))
        self.img_w = int(np.ceil(n_features / self.img_h))
        self.total_size = self.img_h * self.img_w

        self.input_proj = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=5, padding=2),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 3, kernel_size=1),
            nn.BatchNorm2d(3)
        )

        self.resnet = models.resnet50(weights=None)
        in_feat = self.resnet.fc.in_features
        self.resnet.fc = nn.Identity()

        self.use_se = use_se
        if use_se:
            self.se = nn.Sequential(
                nn.Linear(in_feat, in_feat // 16),
                nn.ReLU(inplace=True),
                nn.Linear(in_feat // 16, in_feat),
                nn.Sigmoid()
            )

        self.fc1 = nn.Sequential(nn.Linear(in_feat, 1024), nn.BatchNorm1d(1024), nn.ReLU(inplace=True), nn.Dropout(0.5))
        self.fc2 = nn.Sequential(nn.Linear(1024, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True), nn.Dropout(0.4))
        self.fc3 = nn.Sequential(nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(inplace=True), nn.Dropout(0.3))
        self.fc_out = nn.Linear(256, num_classes)
        self.residual = nn.Linear(in_feat, 256)

    def forward(self, x):
        bs = x.shape[0]
        if self.total_size > self.n_features:
            pad = torch.zeros(bs, self.total_size - self.n_features, device=x.device, dtype=x.dtype)
            x = torch.cat([x, pad], dim=1)
        x = x.view(bs, 1, self.img_h, self.img_w)
        x = self.input_proj(x)
        x = F.relu(x, inplace=True)
        x = F.interpolate(x, size=(224, 224), mode='bilinear', align_corners=False)
        feat = self.resnet(x)
        if self.use_se:
            feat = feat * self.se(feat)
        out = self.fc1(feat)
        out = self.fc2(out)
        out = self.fc3(out)
        out = out + self.residual(feat)
        return self.fc_out(out)

# =============================================================================
# REPLACE CONVOLUTION LAYERS
# =============================================================================
def replace_conv_with_approx(model, mre, config_name, target_path='layer1', max_layers=None):
    """Replace Conv2d layers with ApproxConv2d"""
    count = 0
    replaced_paths = []

    def replace_in_module(module, path=""):
        nonlocal count
        for name, child in module.named_children():
            full_path = f"{path}.{name}" if path else name

            if isinstance(child, nn.Conv2d) and target_path in full_path:
                if max_layers is None or count < max_layers:
                    if 'downsample' in full_path or max_layers is not None:
                        old_weight = child.weight.data.clone()
                        old_bias = child.bias.data.clone() if child.bias is not None else None
                        has_bias = child.bias is not None

                        new_layer = ApproxConv2d(
                            in_channels=child.in_channels,
                            out_channels=child.out_channels,
                            kernel_size=child.kernel_size,
                            stride=child.stride,
                            padding=child.padding,
                            dilation=child.dilation,
                            groups=child.groups,
                            bias=has_bias,
                            mre=mre,
                            axx_mult=config_name
                        )
                        new_layer.weight.data = old_weight
                        if has_bias:
                            new_layer.bias.data = old_bias

                        setattr(module, name, new_layer)
                        count += 1
                        replaced_paths.append(full_path)
            else:
                replace_in_module(child, full_path)

    replace_in_module(model)
    return count, replaced_paths

# =============================================================================
# PSNR CALCULATION
# =============================================================================
def calculate_psnr(accurate, approximate):
    mse = np.mean((accurate - approximate) ** 2)
    if mse == 0:
        return 100.0
    max_val = np.max(np.abs(accurate))
    return 20 * np.log10(max_val / np.sqrt(mse)) if max_val > 0 else 0.0

# =============================================================================
# SHAP PLOTTING
# =============================================================================
def plot_shap_waterfall(shap_values, expected_value, x_explain, feature_names,
                       config_name, pred_class, pred_prob, class_names,
                       accuracy, sample_idx=0, top_n=15):
    shap_values = np.array(shap_values).flatten()

    if np.allclose(shap_values, 0, atol=1e-6) or np.max(np.abs(shap_values)) < 1e-6:
        print(f"  ⚠️  SHAP values near-zero (max={np.max(np.abs(shap_values)):.2e}), skipping {config_name}")
        return

    print(f"  ✓ SHAP valid (range: [{shap_values.min():.4f}, {shap_values.max():.4f}])")

    n_features = len(shap_values)
    top_n = min(top_n, n_features)

    abs_shap = np.abs(shap_values)
    top_idx = np.argsort(abs_shap)[-top_n:][::-1]

    top_explanation = shap.Explanation(
        values=shap_values[top_idx],
        base_values=expected_value,
        data=x_explain[0, top_idx],
        feature_names=[feature_names[i] for i in top_idx]
    )

    plt.figure(figsize=(12, 9))
    shap.plots.waterfall(top_explanation, max_display=top_n, show=False)
    config_level = MAC_CONFIGURATIONS[config_name]['level']
    plt.title(f'SHAP Waterfall - {config_name} (Level {config_level})\n'
              f'Pred: {class_names[pred_class]} ({pred_prob:.1%}) | Acc: {accuracy:.1f}%',
              fontsize=11, fontweight='bold')
    plt.tight_layout()
    filename = f'shap_waterfall_{config_name}_s{sample_idx}.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saved waterfall: {filename}")

    plt.figure(figsize=(12, 8))
    shap.plots.bar(top_explanation, max_display=top_n, show=False)
    plt.title(f'SHAP Bar - {config_name} (Level {config_level}) - Acc: {accuracy:.1f}%',
              fontsize=11, fontweight='bold')
    plt.tight_layout()
    filename = f'shap_bar_{config_name}_s{sample_idx}.png'
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saved bar plot: {filename}")

# =============================================================================
# COMPUTE SHAP
# =============================================================================
def compute_shap_resnet50(model, x_explain, background, feature_names, pred_class):
    def predict_fn(data):
        model.eval()
        with torch.no_grad():
            x_tensor = torch.from_numpy(data.astype(np.float32)).float()
            out = model(x_tensor)
            probs = F.softmax(out, dim=1).cpu().numpy()
            return probs[:, pred_class]

    explainer = shap.KernelExplainer(predict_fn, background, link="identity")
    shap_values = explainer.shap_values(x_explain, nsamples=50, l1_reg="aic")

    if isinstance(shap_values, list):
        shap_values = shap_values[0]

    shap_values = np.array(shap_values).flatten()
    return shap_values, explainer.expected_value

# =============================================================================
# MAIN EXECUTION
# =============================================================================
def main():
    print(f"Configuration:")
    print(f"  Sample: {SAMPLE_IDX}")
    print("="*70 + "\n")

    # Load data
    print("Loading dataset...")
    df = pd.read_csv(DATASET_PATH)
    TARGET_COL = ' Label'
    columns_to_drop = ['Unnamed: 0', ' Label', 'Flow ID', ' Source IP', ' Source Port', ' Destination IP', ' Destination Port', ' Timestamp']
    columns_to_drop = [col for col in columns_to_drop if col in df.columns]
    feature_cols = [col for col in df.columns if col not in columns_to_drop]

    X = df[feature_cols].copy()
    y = df[TARGET_COL].copy()
    del df
    gc.collect()

    for col in X.columns:
        X[col] = pd.to_numeric(X[col], errors='coerce')
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0).astype(np.float32).values

    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(y)
    class_names = label_encoder.classes_

    # Scale
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    X = np.clip(X, -6, 6)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    print(f"✓ Data loaded: {X_train.shape[0]} train, {X_test.shape[0]} test")
    print(f"✓ Number of features: {len(feature_cols)}")

    # Load model info
    with open(MODEL_INFO_PATH, 'r') as f:
        model_info = json.load(f)
    n_features = model_info['n_features']
    num_classes = model_info['num_classes']

    # Load baseline model
    baseline_model = EnhancedResNet50(n_features, num_classes, use_se=True)
    baseline_model.load_state_dict(torch.load(MODEL_PATH, map_location='cpu', weights_only=True))
    baseline_model.eval()
    print("✓ Baseline model loaded")

    # Extract layer descriptors
    print("\nExtracting layer descriptors with forward pass tracing...")
    profiler = LayerProfiler(baseline_model, input_shape=(1, n_features))
    layer_descriptors = profiler.profile()
    print(f"✓ Extracted {len(layer_descriptors)} layers")

    # Show first 10 layers
    print("\nFirst 10 layers:")
    for i, desc in enumerate(layer_descriptors[:10]):
        print(f"  {i:2d}. {desc}")
    if len(layer_descriptors) > 10:
        print(f"  ... ({len(layer_descriptors) - 10} more layers)")

    # Get prediction for sample
    input_data = X_test[SAMPLE_IDX].astype(np.float32)
    with torch.no_grad():
        out = baseline_model(torch.from_numpy(input_data.reshape(1, -1)).float())
        pred_class = out.argmax(1).item()
        pred_prob = F.softmax(out, dim=1)[0, pred_class].item()

    print(f"\nSample {SAMPLE_IDX}:")
    print(f"  Prediction: {class_names[pred_class]} (confidence: {pred_prob:.3f})")
    print(f"  True Label: {class_names[y_test[SAMPLE_IDX]]}")

    # Prepare SHAP background
    benign_idx = np.where(y_train == 0)[0]
    attack_idx = np.where(y_train == 1)[0]
    background = np.vstack([X_train[benign_idx[:4]], X_train[attack_idx[:4]]])
    x_explain = input_data.reshape(1, -1)

    # =============================================================================
    # BASELINE EVALUATION
    # =============================================================================
    print("\n" + "="*70)
    print("BASELINE EVALUATION (mul8s_1KV6_add16se_2TN)")
    print("="*70)

    energy_model_baseline = RoHNASEnergyModel('mul8s_1KV6_add16se_2TN')

    # Calculate baseline energy
    energy_results_baseline = energy_model_baseline.calculate_shap_energy(
        n_features=len(feature_cols),
        n_samples=50,
        layer_descriptors=layer_descriptors
    )

    # Track activations and SAVE to file
    print("\nTracking and saving activation inputs...")
    activation_tracker = ActivationTracker()
    activation_tracker.register_hooks(baseline_model)

    with torch.no_grad():
        test_out = baseline_model(torch.from_numpy(X_test[:200]).float())
        acc_baseline = 100.0 * (test_out.argmax(1) == torch.from_numpy(y_test[:200]).long()).float().mean().item()

    activation_tracker.remove_hooks()

    # SAVE ACTUAL ACTIVATION TENSORS
    baseline_activation_data = activation_tracker.save_activations('activation_inputs_mul8s_1KV6_add16se_2TN.pkl')
    activation_summary_baseline = activation_tracker.get_summary(top_n=10)

    energy_mJ_baseline = energy_results_baseline['shap_energy_mJ']

    print("\n" + "="*80)
    print("BASELINE ENERGY (ROHNAS + EVOAPPROX ANALYTICAL MODEL)")
    print("="*80)
    print(f"Configuration:              {energy_model_baseline.config_name}")
    print(f"SHAP Energy:                {energy_mJ_baseline:.4f} mJ")
    print(f"SHAP Latency:               {energy_results_baseline['shap_latency_ms']:.2f} ms")
    print(f"Energy per forward pass:    {energy_results_baseline['base_energy_per_pass_mJ']:.6f} mJ")
    print(f"Latency per forward pass:   {energy_results_baseline['base_latency_per_pass_ms']:.4f} ms")
    print(f"Total forward passes:       {energy_results_baseline['forward_passes']}")
    print(f"Total memory footprint:     {energy_results_baseline['total_memory_MiB']:.2f} MiB")
    print(f"Accuracy:                   {acc_baseline:.2f}%")
    print("="*80)

    # Print activation summary
    print("\n" + "="*80)
    print("TOP 10 ACTIVATION INPUTS (by absolute mean)")
    print("="*80)
    print(f"{'Layer':<40} {'Mean':<10} {'Std':<10} {'Min':<10} {'Max':<10} {'AbsMean':<10}")
    print("-" * 80)
    for act in activation_summary_baseline:
        print(f"{act['layer']:<40} {act['mean']:<10.4f} {act['std']:<10.4f} {act['min']:<10.4f} {act['max']:<10.4f} {act['abs_mean']:<10.4f}")
    print("="*80)

    # Compute baseline SHAP
    print("\nComputing baseline SHAP...")
    start_time = time.time()
    phi_acc, expected_value = compute_shap_resnet50(baseline_model, x_explain, background,
                                                    feature_cols, pred_class)
    baseline_shap_time = time.time() - start_time
    print(f"✓ Baseline SHAP Time: {baseline_shap_time:.2f}s")

    # Store baseline results
    all_results = {}
    all_results['mul8s_1KV6_add16se_2TN'] = {
        'shap_vals': phi_acc,
        'level': 0,
        'psnr': 100.0,
        'corr': 1.0,
        'energy': energy_mJ_baseline,
        'accuracy': acc_baseline,
        'pred_prob': pred_prob,
        'shap_time': baseline_shap_time,
        'energy_results': energy_results_baseline,
        'mre': 0.0,
        'activation_summary': activation_summary_baseline,
        'activation_file': 'activation_inputs_mul8s_1KV6_add16se_2TN.pkl'
    }

    # Generate baseline plots
    plot_shap_waterfall(phi_acc, expected_value, x_explain, feature_cols,
                       'mul8s_1KV6_add16se_2TN', pred_class, pred_prob,
                       class_names, acc_baseline, SAMPLE_IDX)

    # =============================================================================
    # EVALUATE ALL APPROXIMATE CONFIGURATIONS
    # =============================================================================
    all_configs = list(MAC_CONFIGURATIONS.keys())
    approx_configs = [c for c in all_configs if c != 'mul8s_1KV6_add16se_2TN']

    for config_name in approx_configs:
        print(f"\n{'='*70}")
        print(f"Evaluating {config_name} (Level {MAC_CONFIGURATIONS[config_name]['level']})...")
        print('='*70)

        config = MAC_CONFIGURATIONS[config_name]
        level = config['level']
        mre = config['mre']

        # Create fresh model
        test_model = EnhancedResNet50(n_features, num_classes, use_se=True)
        test_model.load_state_dict(torch.load(MODEL_PATH, map_location='cpu', weights_only=True))

        # Replace layers
        if level > 0:
            replaced, replaced_paths = replace_conv_with_approx(
                test_model, mre=mre, config_name=config_name, target_path='layer1'
            )
            print(f"  ✓ Replaced {replaced} layers")
        test_model.eval()

        # Track activations and SAVE to file
        print(f"\n  Tracking and saving activation inputs for {config_name}...")
        activation_tracker = ActivationTracker()
        activation_tracker.register_hooks(test_model)

        # Test accuracy
        with torch.no_grad():
            test_out = test_model(torch.from_numpy(X_test[:200]).float())
            acc_approx = 100.0 * (test_out.argmax(1) == torch.from_numpy(y_test[:200]).long()).float().mean().item()

        activation_tracker.remove_hooks()

        # SAVE ACTUAL ACTIVATION TENSORS
        activation_file = f'activation_inputs_{config_name}.pkl'
        approx_activation_data = activation_tracker.save_activations(activation_file)
        activation_summary_approx = activation_tracker.get_summary(top_n=10)

        # Energy calculation
        energy_model = RoHNASEnergyModel(config_name)
        test_profiler = LayerProfiler(test_model, input_shape=(1, n_features))
        test_layer_descriptors = test_profiler.profile()

        energy_results = energy_model.calculate_shap_energy(
            n_features=len(feature_cols),
            n_samples=50,
            layer_descriptors=test_layer_descriptors
        )
        energy_mJ = energy_results['shap_energy_mJ']

        # Compute SHAP
        print(f"  Computing SHAP (MRE={mre:.2f}%)...")
        try:
            start_time = time.time()
            phi_apx, expected_val_apx = compute_shap_resnet50(test_model, x_explain, background,
                                                              feature_cols, pred_class)
            approx_shap_time = time.time() - start_time
            phi_apx = np.array(phi_apx).flatten()
            print(f"  ✓ SHAP successful, Time={approx_shap_time:.2f}s")
        except Exception as e:
            print(f"  ⚠️  SHAP failed: {e}")
            phi_apx = np.zeros_like(phi_acc)
            expected_val_apx = expected_value
            approx_shap_time = 0.0

        # Calculate metrics
        psnr_value = calculate_psnr(phi_acc, phi_apx)
        corr_value = spearmanr(phi_acc, phi_apx)[0]
        if np.isnan(corr_value):
            corr_value = 0.0

        energy_savings_pct = ((energy_mJ_baseline - energy_mJ) / energy_mJ_baseline) * 100

        # Store results
        all_results[config_name] = {
            'shap_vals': phi_apx,
            'level': level,
            'psnr': psnr_value,
            'corr': corr_value,
            'energy': energy_mJ,
            'mre': mre,
            'accuracy': acc_approx,
            'pred_prob': pred_prob,
            'shap_time': approx_shap_time,
            'energy_results': energy_results,
            'activation_summary': activation_summary_approx,
            'activation_file': activation_file
        }

        print(f"  Energy:         {energy_mJ:.4f} mJ")
        print(f"  Energy Savings: {energy_savings_pct:.1f}%")
        print(f"  Latency:        {energy_results['shap_latency_ms']:.2f} ms")
        print(f"  Correlation:    {corr_value:.4f}")
        print(f"  PSNR:           {psnr_value:.2f} dB")
        print(f"  Accuracy:       {acc_approx:.2f}%")
        print(f"  SHAP Time:      {approx_shap_time:.2f} s")

        # Print activation summary
        print(f"\n  Top 5 Activation Inputs for {config_name}:")
        print(f"  {'Layer':<40} {'Mean':<10} {'Std':<10} {'AbsMean':<10}")
        print("  " + "-" * 70)
        for act in activation_summary_approx[:5]:
            print(f"  {act['layer']:<40} {act['mean']:<10.4f} {act['std']:<10.4f} {act['abs_mean']:<10.4f}")

        # Generate plots
        plot_shap_waterfall(phi_apx, expected_val_apx, x_explain, feature_cols,
                           config_name, pred_class, pred_prob, class_names,
                           acc_approx, SAMPLE_IDX)

        # Clean up
        del test_model
        gc.collect()

    # =============================================================================
    # RESULTS TABLE
    # =============================================================================
    print("\n" + "="*160)
    print("COMPREHENSIVE RESULTS TABLE (ROHNAS + EVOAPPROX + ACTUAL ACTIVATION INPUTS SAVED)")
    print("="*160)
    print(f"{'Configuration':<30} {'Level':<6} {'Energy(mJ)':<12} {'Savings%':<10} {'Latency(ms)':<12} "
          f"{'Corr':<8} {'PSNR(dB)':<10} {'Acc%':<8} {'Time(s)':<10} {'AvgActMean':<12} {'ActivationFile':<40}")
    print("-" * 160)

    for config_name in sorted(all_results.keys(), key=lambda x: MAC_CONFIGURATIONS[x]['level']):
        result = all_results[config_name]
        savings = ((energy_mJ_baseline - result['energy']) / energy_mJ_baseline) * 100
        latency = result['energy_results']['shap_latency_ms']
        avg_act_mean = np.mean([act['abs_mean'] for act in result['activation_summary']])

        print(f"{config_name:<30} {result['level']:<6} {result['energy']:<12.4f} {savings:<10.1f} {latency:<12.2f} "
              f"{result['corr']:<8.4f} {result['psnr']:<10.2f} {result['accuracy']:<8.2f} {result['shap_time']:<10.2f} "
              f"{avg_act_mean:<12.4f} {result['activation_file']:<40}")

    print("="*160 + "\n")

    # Best configuration
    best_config = max(approx_configs, key=lambda x: all_results[x]['corr'])
    best_result = all_results[best_config]
    best_savings = ((energy_mJ_baseline - best_result['energy']) / energy_mJ_baseline) * 100

    print(f"🏆 Best Configuration (by correlation): {best_config}")
    print(f"   Level:        {best_result['level']}")
    print(f"   Energy:       {best_result['energy']:.4f} mJ")
    print(f"   Savings:      {best_savings:.1f}%")
    print(f"   Latency:      {best_result['energy_results']['shap_latency_ms']:.2f} ms")
    print(f"   Accuracy:     {best_result['accuracy']:.2f}%")
    print(f"   Correlation:  {best_result['corr']:.4f}")
    print(f"   PSNR:         {best_result['psnr']:.2f} dB")
    print(f"   Activation File: {best_result['activation_file']}")

    print("\n" + "="*70)
    print("SAVED ACTIVATION INPUT FILES:")
    print("="*70)
    for config_name in sorted(all_results.keys(), key=lambda x: MAC_CONFIGURATIONS[x]['level']):
        print(f"  {all_results[config_name]['activation_file']}")
    print("="*70)

    return all_results

if __name__ == "__main__":
    try:
        results = main()
        print("\n✅ COMPLETED - All configurations evaluated with actual activation inputs saved")
        print("\nTo load activation inputs:")
        print("  with open('activation_inputs_mul8s_1KV6_add16se_2TN.pkl', 'rb') as f:")
        print("      activations = pickle.load(f)")
        print("  # Access actual tensor values:")
        print("  for layer_name, activation_array in activations.items():")
        print("      print(f'{layer_name}: {activation_array.shape}')")
    except Exception as e:
        print(f"\n❌ Error: {e}")
        import traceback
        traceback.print_exc()

DDoS ResNet50 SHAP Analysis - ALL MULTIPLIER CONFIGURATIONS
WITH ROHNAS ANALYTICAL ENERGY MODEL
Configuration:
  Sample: 0

Loading dataset...
✓ Data loaded: 19211 train, 4803 test
✓ Number of features: 80
✓ Baseline model loaded

Extracting layer descriptors with forward pass tracing...
✓ Extracted 64 layers

First 10 layers:
   0. Conv(input_proj.0): [1×9×9] → [32×9×9], k=5
   1. Conv(input_proj.3): [32×9×9] → [32×9×9], k=3
   2. Conv(input_proj.6): [32×9×9] → [16×9×9], k=3
   3. Conv(input_proj.9): [16×9×9] → [3×9×9], k=1
   4. Conv(resnet.conv1): [3×224×224] → [64×112×112], k=7
   5. Conv(resnet.layer1.0.conv1): [64×56×56] → [64×56×56], k=1
   6. Conv(resnet.layer1.0.conv2): [64×56×56] → [64×56×56], k=3
   7. Conv(resnet.layer1.0.conv3): [64×56×56] → [256×56×56], k=1
   8. Conv(resnet.layer1.0.downsample.0): [64×56×56] → [256×56×56], k=1
   9. Conv(resnet.layer1.1.conv1): [256×56×56] → [64×56×56], k=1
  ... (54 more layers)

Sample 0:
  Prediction: DrDoS_DNS (confidence: 0.635)
  T

  0%|          | 0/1 [00:00<?, ?it/s]

✓ Baseline SHAP Time: 25.31s
  ✓ SHAP valid (range: [0.0000, 0.0617])
  ✓ Saved waterfall: shap_waterfall_mul8s_1KV6_add16se_2TN_s0.png
  ✓ Saved bar plot: shap_bar_mul8s_1KV6_add16se_2TN_s0.png

Evaluating mul8s_1KV8_add16se_2UY (Level 1)...
  ✓ Replaced 1 layers

RoHNAS Energy Model with EvoApprox MAC: mul8s_1KV8_add16se_2UY
Hardware Parameters:
  Clock Period (T):        3.0 ns
  PE Array Size:           256×256
  Memory Energy (SRAM):    5.0 pJ/access (CACTI-P @ 45nm)

EvoApprox MAC Configuration:
  Approximation Level:     1
  MRE:                     0.28%
  Multiplier Energy:       0.422 pJ
  Adder Energy:            0.071 pJ
  Total MAC Energy (E_MAC): 0.493 pJ

Energy Model Formula:
  E_total = Σ(m_acc/128 × E_memory) + Σ(cl × E_MAC)
  ↳ Replaced fixed PE power with EvoApprox MAC energy

  Computing SHAP (MRE=0.28%)...


  0%|          | 0/1 [00:00<?, ?it/s]

  ✓ SHAP successful, Time=25.80s
  Energy:         2133.2797 mJ
  Energy Savings: 0.8%
  Latency:        12955786.37 ms
  Correlation:    0.1624
  PSNR:           15.71 dB
  Accuracy:       95.00%
  SHAP Time:      25.80 s
  ✓ SHAP valid (range: [-0.0037, 0.0326])
  ✓ Saved waterfall: shap_waterfall_mul8s_1KV8_add16se_2UY_s0.png
  ✓ Saved bar plot: shap_bar_mul8s_1KV8_add16se_2UY_s0.png

Evaluating mul8s_1KV9_add16se_2U6 (Level 2)...
  ✓ Replaced 1 layers

RoHNAS Energy Model with EvoApprox MAC: mul8s_1KV9_add16se_2U6
Hardware Parameters:
  Clock Period (T):        3.0 ns
  PE Array Size:           256×256
  Memory Energy (SRAM):    5.0 pJ/access (CACTI-P @ 45nm)

EvoApprox MAC Configuration:
  Approximation Level:     2
  MRE:                     0.9%
  Multiplier Energy:       0.410 pJ
  Adder Energy:            0.066 pJ
  Total MAC Energy (E_MAC): 0.476 pJ

Energy Model Formula:
  E_total = Σ(m_acc/128 × E_memory) + Σ(cl × E_MAC)
  ↳ Replaced fixed PE power with EvoApprox MAC energy

...

  ✓ SHAP successful, Time=27.80s
  Energy:         1092.4982 mJ
  Energy Savings: 49.2%
  Latency:        12955786.37 ms
  Correlation:    -0.0512
  PSNR:           15.48 dB
  Accuracy:       92.50%
  SHAP Time:      27.80 s
  ✓ SHAP valid (range: [-0.0023, 0.0314])
  ✓ Saved waterfall: shap_waterfall_mul8s_1L2L_add16se_2YM_s0.png
  ✓ Saved bar plot: shap_bar_mul8s_1L2L_add16se_2YM_s0.png

Evaluating mul8s_1L2N_add16se_32T (Level 6)...
  ✓ Replaced 1 layers

RoHNAS Energy Model with EvoApprox MAC: mul8s_1L2N_add16se_32T
Hardware Parameters:
  Clock Period (T):        3.0 ns
  PE Array Size:           256×256
  Memory Energy (SRAM):    5.0 pJ/access (CACTI-P @ 45nm)

EvoApprox MAC Configuration:
  Approximation Level:     6
  MRE:                     27.44%
  Multiplier Energy:       0.126 pJ
  Adder Energy:            0.048 pJ
  Total MAC Energy (E_MAC): 0.174 pJ

Energy Model Formula:
  E_total = Σ(m_acc/128 × E_memory) + Σ(cl × E_MAC)
  ↳ Replaced fixed PE power with EvoApprox MAC en

  0%|          | 0/1 [00:00<?, ?it/s]

  ✓ SHAP successful, Time=27.84s
  Energy:         755.6478 mJ
  Energy Savings: 64.9%
  Latency:        12955786.37 ms
  Correlation:    0.0645
  PSNR:           16.25 dB
  Accuracy:       91.00%
  SHAP Time:      27.84 s
  ✓ SHAP valid (range: [-0.0028, 0.0299])
  ✓ Saved waterfall: shap_waterfall_mul8s_1L2N_add16se_32T_s0.png
  ✓ Saved bar plot: shap_bar_mul8s_1L2N_add16se_32T_s0.png

Evaluating mul8s_1L12_add16se_3BD (Level 7)...
  ✓ Replaced 1 layers

RoHNAS Energy Model with EvoApprox MAC: mul8s_1L12_add16se_3BD
Hardware Parameters:
  Clock Period (T):        3.0 ns
  PE Array Size:           256×256
  Memory Energy (SRAM):    5.0 pJ/access (CACTI-P @ 45nm)

EvoApprox MAC Configuration:
  Approximation Level:     7
  MRE:                     135.77%
  Multiplier Energy:       0.052 pJ
  Adder Energy:            0.043 pJ
  Total MAC Energy (E_MAC): 0.095 pJ

Energy Model Formula:
  E_total = Σ(m_acc/128 × E_memory) + Σ(cl × E_MAC)
  ↳ Replaced fixed PE power with EvoApprox MAC ene

  0%|          | 0/1 [00:00<?, ?it/s]

  ✓ SHAP successful, Time=27.81s
  Energy:         414.4787 mJ
  Energy Savings: 80.7%
  Latency:        12955786.37 ms
  Correlation:    0.1334
  PSNR:           15.57 dB
  Accuracy:       92.00%
  SHAP Time:      27.81 s
  ✓ SHAP valid (range: [0.0000, 0.0165])
  ✓ Saved waterfall: shap_waterfall_mul8s_1L12_add16se_3BD_s0.png
  ✓ Saved bar plot: shap_bar_mul8s_1L12_add16se_3BD_s0.png

COMPREHENSIVE RESULTS TABLE (ROHNAS + EVOAPPROX ANALYTICAL MODEL)
Configuration                  Level  Energy(mJ)   Savings%   Latency(ms)  Corr     PSNR(dB)   Acc%     Time(s)   
------------------------------------------------------------------------------------------------------------------------
mul8s_1KV6_add16se_2TN         0      2150.5541    0.0        12955786.37  1.0000   100.00     95.00    25.31     
mul8s_1KV8_add16se_2UY         1      2133.2797    0.8        12955786.37  0.1624   15.71      95.00    25.80     
mul8s_1KV9_add16se_2U6         2      2059.8636    4.2        12955786.37  0.0